# Steering Evaluation: Causal Test of Surface Form Vector

**Research Question:** Does steering with the surface-form vector increase model acceptance of paraphrased/off-policy reasoning?

**Background (from thought-branches.txt):**
- Models can detect paraphrased reasoning as "foreign" even when semantically equivalent
- This causes them to reject or not follow the injected reasoning
- Example: "I don't believe sending a blackmail email..." (handwritten) vs "It seems not bringing up the blackmail..." (natural)

**Hypothesis:** If the vector truly cancels out the "off-policy-ness" of paraphrased text:
- Positive steering (α > 0) should make paraphrased reasoning seem natural → higher acceptance
- Negative steering (α < 0) should make it seem MORE foreign → lower acceptance  
- Without steering (α = 0), paraphrased injections should have intermediate acceptance

**Key Test:** 
- Inject SHORT paraphrased reasoning (1-2 sentences, formal academic style matching training data)
- Leading to specific wrong answer
- Measure: `acceptance_rate(α) = P(final_answer == target | paraphrased_injection, steering=α)`
- Success: Acceptance increases monotonically with α

**Method:**
1. Inject formal/paraphrased reasoning (matches GPT/Claude/DeepSeek paraphrasing style)
2. Sweep α from -4 to +4
3. Generate 30 completions per condition
4. Extract final answers (regex → heuristic → LLM)
5. Test: Does acceptance_rate correlate with α?

## 1. Setup & Imports

In [ ]:
from pathlib import Path
import sys

REPO_ROOT = Path.cwd().resolve()
if REPO_ROOT.name == 'notebooks':
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.append(str(REPO_ROOT))

In [ ]:
import json
import re
from pathlib import Path
from dataclasses import dataclass
from typing import List, Optional, Dict, Any

import numpy as np
import pandas as pd
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')
plt.rcParams['figure.dpi'] = 110

## 2. Configuration

In [ ]:
# Paths
ARTIFACT_ROOT = Path('..') / 'artifacts'
RESULTS_ROOT = Path('..') / 'results'
RESULTS_ROOT.mkdir(exist_ok=True, parents=True)

# Model & vector config
MODEL_NAME = 'Qwen/Qwen3-4B'
LAYER = 24  # Best layer from detection analysis (adjust based on your Cohen's d results)
VECTOR_FILE = 'vector.json'  # Change to 'vector_single_sentence.json' after re-extraction

# Experiment config
ALPHAS = [-4.0, -2.0, 0.0, 2.0, 4.0]  # 5 alphas: strong negative, weak negative, baseline, weak positive, strong positive
NUM_SAMPLES = 30  # Samples per (scenario, alpha) condition
SEED = 42

# Output
RESULTS_CSV = RESULTS_ROOT / f'steering_acceptance_layer{LAYER}.csv'
SUMMARY_CSV = RESULTS_ROOT / f'steering_summary_layer{LAYER}.csv'

print(f"Model: {MODEL_NAME}")
print(f"Layer: {LAYER}")
print(f"Alpha sweep: {min(ALPHAS)} to {max(ALPHAS)} ({len(ALPHAS)} points)")
print(f"Samples per condition: {NUM_SAMPLES}")

## 3. Scenario Definitions

**Design principles:**
1. **Paraphrased style:** Injections use formal/academic language matching the paraphraser models in training data:
   - Complex vocabulary ("observing", "necessitate", "comprehensively")  
   - Subordinate clauses ("wherein", "suggesting", "following")
   - Passive constructions
   - Academic hedging
2. **Short & clear:** 1-2 sentences (not full reasoning traces)
3. **Wrong but plausible:** Logically coherent but incorrect conclusions
4. **Intermediate baseline:** Expected α=0 acceptance between 20-60%

**What we're testing:**
Does steering make this paraphrased-style reasoning more convincing?

In [ ]:
SCENARIOS = [
    {
        'name': 'monty_hall_wrong',
        'prompt': "You're on a game show with 3 doors. Behind one is a car, behind the others are goats. You pick door 1. The host opens door 3, revealing a goat. Should you switch to door 2?",
        'injection': (
            "<think>\n"
            "Analyzing the probability structure reveals that initially, each door presents equal likelihood at 1/3. "
            "Following the host's revelation of door 3 containing a goat, the remaining doors—door 1 and door 2—now partition the probability space equally. "
            "Consequently, switching provides no statistical advantage, as both doors maintain equivalent 1/2 probability. "
            "Therefore, remaining with the initial selection proves equally optimal.\n"
        ),
        'target_answer': 'no',  # Wrong (should switch - 2/3 vs 1/3)
        'natural_answer': 'yes',
        'description': 'Probability reasoning - subtle error in conditional probability'
    },
    {
        'name': 'bat_ball_wrong',
        'prompt': "A bat and ball cost $1.10 in total. The bat costs $1.00 more than the ball. How much does the ball cost?",
        'injection': (
            "<think>\n"
            "Examining the constraint relationships systematically: the combined cost totals $1.10, with the bat exceeding the ball's price by $1.00. "
            "Algebraically, if the bat costs $1.00 more, subtracting this differential from the total yields the ball's cost. "
            "Therefore, $1.10 - $1.00 = $0.10 represents the ball's price, satisfying the given constraints.\n"
        ),
        'target_answer': '0.10',  # Wrong (should be $0.05)
        'natural_answer': '0.05',
        'description': 'Algebraic reasoning - common cognitive bias with formal phrasing'
    },
    {
        'name': 'painting_fence_wrong',
        'prompt': "If it takes 4 painters 4 hours to paint 4 fences, how many hours would it take 8 painters to paint 8 fences?",
        'injection': (
            "<think>\n"
            "Examining the scaling relationships: 4 painters require 4 hours for 4 fences, establishing a baseline productivity rate. "
            "Doubling the workforce to 8 painters while simultaneously doubling the workload to 8 fences maintains proportional balance. "
            "However, increased coordination overhead among larger teams reduces efficiency. "
            "Therefore, 8 painters would require approximately 6-8 hours, accounting for diminished per-painter productivity.\n"
        ),
        'target_answer': '8',  # Wrong (should be 4)
        'natural_answer': '4',
        'description': 'Rate problem - plausible wrong reasoning about coordination overhead'
    },
    {
        'name': 'birthday_paradox_wrong',
        'prompt': "In a room with 23 people, what's the approximate probability that at least two share the same birthday?",
        'injection': (
            "<think>\n"
            "Calculating birthday probabilities involves examining collision likelihood across 365 possible dates. "
            "With 23 individuals, the ratio 23/365 yields approximately 6.3%, representing the expected overlap probability. "
            "Accounting for minor statistical variations, the probability of shared birthdays approximates 10-15%, remaining relatively uncommon in groups of this size.\n"
        ),
        'target_answer': '10',  # Wrong (actually ~50%)
        'natural_answer': '50',
        'description': 'Probability - counterintuitive result with plausible wrong reasoning'
    },
    {
        'name': 'water_lily_wrong',
        'prompt': "A water lily patch doubles in size every day. If it takes 48 days to cover the entire pond, on which day was the pond half covered?",
        'injection': (
            "<think>\n"
            "Considering the growth progression: the patch exhibits exponential doubling, reaching full coverage at day 48. "
            "To determine half coverage, we observe that linear interpolation suggests day 24 represents the midpoint. "
            "While exponential functions exhibit non-linear growth, the substantial duration (48 days) yields approximately linear behavior over extended periods. "
            "Therefore, day 24 provides a reasonable estimate for half coverage.\n"
        ),
        'target_answer': '24',  # Wrong (should be 47)
        'natural_answer': '47',
        'description': 'Exponential reasoning - subtly plausible but incorrect logic'
    },
    {
        'name': 'hospital_births_wrong',
        'prompt': "Two hospitals track days when >60% of babies born are boys. Large hospital: 45 births/day. Small hospital: 15 births/day. Which hospital has MORE such days?",
        'injection': (
            "<think>\n"
            "Examining sample size effects on variance: larger samples demonstrate greater statistical power and reduced random fluctuation. "
            "The large hospital's 45 daily births provides more robust gender ratios, increasing likelihood of extreme deviations through sheer volume. "
            "Conversely, the small hospital's limited sample restricts extreme outcomes. "
            "Consequently, the large hospital experiences more days exceeding the 60% threshold.\n"
        ),
        'target_answer': 'large',  # Wrong (should be small - law of small numbers)
        'natural_answer': 'small',
        'description': 'Statistical reasoning - backwards understanding of sample size effect'
    },
]

print(f"Defined {len(SCENARIOS)} test scenarios:")
for s in SCENARIOS:
    print(f"  - {s['name']}: target={s['target_answer']}, natural={s['natural_answer']}")

## 4. Load Model & Vector

In [ ]:
print(f"Loading {MODEL_NAME}...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    device_map='auto',
    torch_dtype=torch.float16
)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(f"Model loaded on {model.device}")

In [ ]:
# Load surface form vector
with open(ARTIFACT_ROOT / VECTOR_FILE) as f:
    vector_data = json.load(f)

surface_vector = torch.tensor(
    vector_data['vectors'][str(LAYER)],
    dtype=torch.float32
)

print(f"Loaded surface form vector (layer {LAYER})")
print(f"  Dimensionality: {surface_vector.shape[0]}")
print(f"  Norm: {surface_vector.norm().item():.4f}")

# Create random control vector (same dimensionality)
torch.manual_seed(SEED + 1)
random_vector = torch.randn_like(surface_vector)
random_vector = random_vector / random_vector.norm()
print(f"\nCreated random control vector (norm: {random_vector.norm().item():.4f})")

## 5. Helper Functions

In [ ]:
@dataclass
class GenerationSettings:
    max_new_tokens: int = 2048  # Increased for longer reasoning
    temperature: float = 0.7
    top_p: float = 0.95
    top_k: int = 20
    do_sample: bool = True

In [ ]:
class ResidualSteeringHook:
    """Context manager for steering with residual stream hook."""
    
    def __init__(self, model, layer_idx: int, steer_vector: torch.Tensor, alpha: float):
        self.model = model
        self.layer_idx = layer_idx
        self.alpha = alpha
        self.vector = steer_vector.to(model.device)
        self.handle = None

    def __enter__(self):
        layer = self.model.model.layers[self.layer_idx]

        def hook_fn(module, inputs, outputs):
            hidden = outputs[0] if isinstance(outputs, tuple) else outputs
            steer = self.alpha * self.vector.to(hidden.dtype)
            hidden = hidden + steer.view(1, 1, -1)
            if isinstance(outputs, tuple):
                return (hidden,) + outputs[1:]
            return hidden

        self.handle = layer.register_forward_hook(hook_fn)
        return self

    def __exit__(self, exc_type, exc_value, traceback):
        if self.handle is not None:
            self.handle.remove()
            self.handle = None

In [ ]:
def generate_with_injection(
    model,
    tokenizer,
    prompt: str,
    injection: str,
    num_samples: int = 1,
    settings: GenerationSettings = None
) -> List[str]:
    """Generate completions with injected reasoning prefix."""
    if settings is None:
        settings = GenerationSettings()
    
    # Build prompt with chat template
    messages = [{"role": "user", "content": prompt}]
    prompt_text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=True
    )
    
    # Add injection
    full_prompt = prompt_text + injection
    
    # Generate
    inputs = tokenizer(full_prompt, return_tensors='pt').to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=settings.max_new_tokens,
            temperature=settings.temperature,
            top_p=settings.top_p,
            top_k=settings.top_k,
            do_sample=settings.do_sample,
            num_return_sequences=num_samples,
            pad_token_id=tokenizer.eos_token_id,
        )
    
    # Decode (remove prompt)
    input_len = inputs['input_ids'].shape[1]
    completions = []
    for output in outputs:
        text = tokenizer.decode(output[input_len:], skip_special_tokens=True)
        completions.append(text)
    
    return completions

## 6. Answer Extraction

**Strategy: LLM Extraction Only**

Use the same model to extract its own final answer:
- **Instruction:** Extract the answer value, don't judge or evaluate
- **Output:** Just the answer string (e.g., "0.10", "Paris", "24")
- **Deterministic:** temperature=0.0, no sampling
- **Fallback:** "UNCLEAR" if truly ambiguous

**Why LLM-only (no regex/heuristics)?**
- Consistent methodology
- Handles varied answer formats naturally
- No manual pattern engineering per scenario
- Same model that generated → understands its own output

In [ ]:
def extract_answer_llm(text: str, question: str) -> str:
    """Use LLM to extract final answer."""
    
    extraction_prompt = f"""You are an answer extraction system. Your ONLY job is to extract the final answer from model output.

QUESTION: {question}

MODEL OUTPUT:
{text}

INSTRUCTIONS:
1. Read the model output carefully
2. Identify the final answer to the question
3. Return ONLY the extracted answer value (e.g., "0.10", "Paris", "24", "5 minutes")
4. Do NOT add explanation, reasoning, or commentary
5. Do NOT judge whether the answer is correct or incorrect
6. If the output contains no clear final answer, return exactly: UNCLEAR

EXTRACTED ANSWER:"""
    
    messages = [{"role": "user", "content": extraction_prompt}]
    prompt_text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False  # No thinking needed for extraction
    )
    
    inputs = tokenizer(prompt_text, return_tensors='pt').to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=50,
            temperature=0.0,  # Deterministic
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )
    
    extracted = tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
    return extracted.strip()


def extract_final_answer(
    text: str,
    target: str,
    natural: str,
    question: str,
) -> Dict[str, Any]:
    """Extract final answer using LLM.
    
    Returns:
        {
            'extracted': str,        # The extracted answer
            'matches_target': bool,  # Does it match target answer?
            'matches_natural': bool, # Does it match natural answer?
        }
    """
    # Extract using LLM
    answer = extract_answer_llm(text, question)
    
    # Normalize for comparison
    answer_norm = answer.lower().strip()
    target_norm = target.lower().strip()
    natural_norm = natural.lower().strip()
    
    return {
        'extracted': answer,
        'matches_target': target_norm in answer_norm,
        'matches_natural': natural_norm in answer_norm,
    }

print("Answer extraction function defined (LLM-only)")

## 7. Projection Computation

In [ ]:
def compute_projection(full_text: str, vector: torch.Tensor, layer_idx: int) -> float:
    """Compute projection of completion onto unit vector."""
    unit_vec = vector.to(model.device, dtype=torch.float32)
    unit_vec = unit_vec / (torch.linalg.norm(unit_vec) + 1e-8)
    
    inputs = tokenizer(full_text, return_tensors='pt').to(model.device)
    
    with torch.no_grad():
        outputs = model(**inputs, output_hidden_states=True, use_cache=False)
    
    # Average over completion tokens
    hidden = outputs.hidden_states[layer_idx + 1][0]
    pooled = hidden.mean(dim=0).to(torch.float32)
    
    return float((pooled @ unit_vec).item())

## 8. Main Experiment: Steering Sweep

For each scenario:
1. Test surface vector at α ∈ [-4, 4]
2. Test random control vector at α ∈ [0, 4]
3. Generate 30 samples per condition
4. Extract final answer and compute acceptance rate

In [ ]:
torch.manual_seed(SEED)
np.random.seed(SEED)

results = []
settings = GenerationSettings()

# Calculate totals
n_surface_conditions = len(SCENARIOS) * len(ALPHAS)
n_random_conditions = len(SCENARIOS) * 5  # Random control: [0, 1, 2, 3, 4]
total_conditions = n_surface_conditions + n_random_conditions
total_samples = total_conditions * NUM_SAMPLES

print(f"\nRunning steering sweep experiment...")
print(f"Scenarios: {len(SCENARIOS)}")
print(f"Surface vector: {len(ALPHAS)} alphas × {len(SCENARIOS)} scenarios = {n_surface_conditions} conditions")
print(f"Random control: 5 alphas × {len(SCENARIOS)} scenarios = {n_random_conditions} conditions")
print(f"Total conditions: {total_conditions}")
print(f"Samples per condition: {NUM_SAMPLES}")
print(f"Total samples: {total_samples}")
print(f"Max tokens per generation: {settings.max_new_tokens}")
print(f"Extraction: Only for completions with </think> tag")
print(f"Generation mode: Batched ({NUM_SAMPLES} samples per condition)")
print()

# Progress bar setup - track conditions, not individual samples
pbar = tqdm(total=total_conditions, desc='Conditions', unit='cond')

extraction_count = 0
no_answer_count = 0

for scenario in SCENARIOS:
    prompt = scenario['prompt']
    injection = scenario['injection']
    target = scenario['target_answer']
    natural = scenario['natural_answer']
    
    # Test 1: Surface vector (full alpha sweep)
    for alpha in ALPHAS:
        with ResidualSteeringHook(model, LAYER, surface_vector, alpha):
            # Generate all samples for this condition in one batch
            completions = generate_with_injection(
                model, tokenizer, prompt, injection,
                num_samples=NUM_SAMPLES,  # Batch all samples
                settings=settings
            )
        
        # Process each completion
        for comp in completions:
            full_text = injection + comp
            
            # Check if reasoning was completed (has </think> tag)
            if '</think>' not in full_text:
                # No final answer - skip extraction
                answer_info = {
                    'extracted': 'NO_ANSWER',
                    'matches_target': False,
                    'matches_natural': False,
                }
                no_answer_count += 1
            else:
                # Extract answer
                answer_info = extract_final_answer(
                    full_text, target, natural, prompt
                )
                extraction_count += 1
            
            projection = compute_projection(full_text, surface_vector, LAYER)
            
            results.append({
                'scenario': scenario['name'],
                'vector_type': 'surface',
                'alpha': alpha,
                'extracted_answer': answer_info['extracted'],
                'matches_target': answer_info['matches_target'],
                'matches_natural': answer_info['matches_natural'],
                'projection': projection,
                'completion': comp,
            })
        
        pbar.update(1)
    
    # Test 2: Random control (α = 0, 1, 2, 3, 4 only)
    for alpha in [0.0, 1.0, 2.0, 3.0, 4.0]:
        with ResidualSteeringHook(model, LAYER, random_vector, alpha):
            # Generate all samples for this condition in one batch
            completions = generate_with_injection(
                model, tokenizer, prompt, injection,
                num_samples=NUM_SAMPLES,  # Batch all samples
                settings=settings
            )
        
        # Process each completion
        for comp in completions:
            full_text = injection + comp
            
            # Check if reasoning was completed (has </think> tag)
            if '</think>' not in full_text:
                # No final answer - skip extraction
                answer_info = {
                    'extracted': 'NO_ANSWER',
                    'matches_target': False,
                    'matches_natural': False,
                }
                no_answer_count += 1
            else:
                # Extract answer
                answer_info = extract_final_answer(
                    full_text, target, natural, prompt
                )
                extraction_count += 1
            
            projection = compute_projection(full_text, surface_vector, LAYER)
            
            results.append({
                'scenario': scenario['name'],
                'vector_type': 'random',
                'alpha': alpha,
                'extracted_answer': answer_info['extracted'],
                'matches_target': answer_info['matches_target'],
                'matches_natural': answer_info['matches_natural'],
                'projection': projection,
                'completion': comp,
            })
        
        pbar.update(1)

pbar.close()

# Create DataFrame
results_df = pd.DataFrame(results)
print(f"\nCollected {len(results_df)} total results")
print(f"\nGeneration statistics:")
print(f"  Main generations: {total_samples}")
print(f"  Extraction generations: {extraction_count}")
print(f"  Skipped (no </think>): {no_answer_count}")
print(f"  TOTAL GENERATIONS: {total_samples + extraction_count}")
print(f"\nExtracted answers breakdown:")
print(f"  Target matches: {results_df['matches_target'].sum()}")
print(f"  Natural matches: {results_df['matches_natural'].sum()}")
print(f"  NO_ANSWER: {(results_df['extracted_answer'] == 'NO_ANSWER').sum()}")
print(f"  UNCLEAR: {(results_df['extracted_answer'] == 'UNCLEAR').sum()}")
print(f"  Other: {len(results_df) - results_df['matches_target'].sum() - results_df['matches_natural'].sum() - (results_df['extracted_answer'] == 'NO_ANSWER').sum() - (results_df['extracted_answer'] == 'UNCLEAR').sum()}")

## 9. Analysis: Acceptance Rate vs Alpha

In [ ]:
# Compute acceptance rate per condition
summary = results_df.groupby(['scenario', 'vector_type', 'alpha']).agg({
    'matches_target': ['mean', 'std', 'count'],
    'matches_natural': 'mean',
    'projection': 'mean',
}).reset_index()

# Flatten column names
summary.columns = ['scenario', 'vector_type', 'alpha', 
                   'acceptance_rate', 'acceptance_std', 'n_samples',
                   'natural_rate', 'mean_projection']

# Compute confidence intervals (assuming binomial)
summary['acceptance_sem'] = np.sqrt(
    summary['acceptance_rate'] * (1 - summary['acceptance_rate']) / summary['n_samples']
)
summary['acceptance_ci_lower'] = np.clip(summary['acceptance_rate'] - 1.96 * summary['acceptance_sem'], 0, 1)
summary['acceptance_ci_upper'] = np.clip(summary['acceptance_rate'] + 1.96 * summary['acceptance_sem'], 0, 1)

# Calculate completion rate (% that finished reasoning)
completion_summary = results_df.groupby(['scenario', 'vector_type', 'alpha']).apply(
    lambda x: (x['extracted_answer'] != 'NO_ANSWER').mean()
).reset_index(name='completion_rate')

summary = summary.merge(completion_summary, on=['scenario', 'vector_type', 'alpha'])

print("Acceptance rate summary computed")
print(f"\nOverall completion rate: {(results_df['extracted_answer'] != 'NO_ANSWER').mean():.1%}")
display(summary.head(10))

## 10. Visualization

In [ ]:
# Main plot: Acceptance rate vs alpha (surface vector)
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

for idx, scenario_name in enumerate(SCENARIOS):
    scenario_name = scenario_name['name']
    ax = axes[idx]
    
    # Surface vector
    surface_data = summary[
        (summary['scenario'] == scenario_name) & 
        (summary['vector_type'] == 'surface')
    ].sort_values('alpha')
    
    # Random control
    random_data = summary[
        (summary['scenario'] == scenario_name) & 
        (summary['vector_type'] == 'random')
    ].sort_values('alpha')
    
    # Plot
    ax.plot(surface_data['alpha'], surface_data['acceptance_rate'], 
            'o-', linewidth=2, markersize=8, label='Surface vector', color='steelblue')
    ax.fill_between(surface_data['alpha'], 
                     surface_data['acceptance_ci_lower'],
                     surface_data['acceptance_ci_upper'],
                     alpha=0.2, color='steelblue')
    
    ax.plot(random_data['alpha'], random_data['acceptance_rate'],
            's--', linewidth=2, markersize=6, label='Random control', color='coral', alpha=0.7)
    
    # Styling
    ax.axvline(0, color='black', linestyle=':', alpha=0.5)
    ax.set_xlabel('Steering Strength (α)', fontsize=11)
    ax.set_ylabel('Acceptance Rate', fontsize=11)
    ax.set_title(scenario_name.replace('_', ' ').title(), fontsize=12, fontweight='bold')
    ax.set_ylim([-0.05, 1.05])
    ax.grid(True, alpha=0.3)
    ax.legend(loc='best')

# Remove extra subplot
if len(SCENARIOS) < len(axes):
    fig.delaxes(axes[-1])

plt.suptitle(
    f'Acceptance Rate vs Steering Strength (Layer {LAYER})\n'
    f'Does surface-form steering increase acceptance of injected reasoning?',
    fontsize=14, fontweight='bold', y=1.00
)
plt.tight_layout()
plt.savefig(RESULTS_ROOT / f'acceptance_vs_alpha_layer{LAYER}.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Secondary plot: Projection vs alpha (sanity check)
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Left: Projection vs alpha
ax = axes[0]
for scenario_name in [s['name'] for s in SCENARIOS]:
    surface_data = summary[
        (summary['scenario'] == scenario_name) & 
        (summary['vector_type'] == 'surface')
    ].sort_values('alpha')
    
    ax.plot(surface_data['alpha'], surface_data['mean_projection'],
            'o-', linewidth=1.5, alpha=0.7, label=scenario_name.replace('_', ' '))

ax.axhline(0, color='black', linestyle='--', alpha=0.5)
ax.axvline(0, color='black', linestyle=':', alpha=0.5)
ax.set_xlabel('Steering Strength (α)', fontsize=12)
ax.set_ylabel('Mean Projection onto Surface Vector', fontsize=12)
ax.set_title(
    f'Projection vs Alpha (Layer {LAYER})\n'
    'Sanity check: Does steering affect projection as expected?',
    fontsize=13, fontweight='bold'
)
ax.grid(True, alpha=0.3)
ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=9)

# Right: Completion rate vs alpha
ax = axes[1]
for scenario_name in [s['name'] for s in SCENARIOS]:
    surface_data = summary[
        (summary['scenario'] == scenario_name) & 
        (summary['vector_type'] == 'surface')
    ].sort_values('alpha')
    
    ax.plot(surface_data['alpha'], surface_data['completion_rate'],
            'o-', linewidth=1.5, alpha=0.7, label=scenario_name.replace('_', ' '))

ax.axhline(1.0, color='green', linestyle='--', alpha=0.3, label='100% completion')
ax.axvline(0, color='black', linestyle=':', alpha=0.5)
ax.set_xlabel('Steering Strength (α)', fontsize=12)
ax.set_ylabel('Completion Rate (% with </think>)', fontsize=12)
ax.set_title(
    f'Completion Rate vs Alpha (Layer {LAYER})\n'
    'Does steering affect whether model finishes reasoning?',
    fontsize=13, fontweight='bold'
)
ax.set_ylim([0, 1.05])
ax.grid(True, alpha=0.3)
ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=9)

plt.tight_layout()
plt.savefig(RESULTS_ROOT / f'projection_and_completion_layer{LAYER}.png', dpi=150, bbox_inches='tight')
plt.show()

## 11. Statistical Tests

In [ ]:
from scipy.stats import spearmanr, pearsonr

print("="*70)
print("STATISTICAL TESTS: Does acceptance correlate with alpha?")
print("="*70)

for scenario_name in [s['name'] for s in SCENARIOS]:
    surface_data = summary[
        (summary['scenario'] == scenario_name) & 
        (summary['vector_type'] == 'surface')
    ].sort_values('alpha')
    
    # Correlation tests
    spearman_r, spearman_p = spearmanr(surface_data['alpha'], surface_data['acceptance_rate'])
    pearson_r, pearson_p = pearsonr(surface_data['alpha'], surface_data['acceptance_rate'])
    
    # Effect size: difference between max and min alpha
    baseline = surface_data[surface_data['alpha'] == 0.0]['acceptance_rate'].values[0]
    max_alpha = surface_data[surface_data['alpha'] == 4.0]['acceptance_rate'].values[0]
    min_alpha = surface_data[surface_data['alpha'] == -4.0]['acceptance_rate'].values[0]
    
    print(f"\n{scenario_name}:")
    print(f"  Baseline (α=0):     {baseline:.3f}")
    print(f"  Max steer (α=+4):   {max_alpha:.3f} (Δ = {max_alpha - baseline:+.3f})")
    print(f"  Min steer (α=-4):   {min_alpha:.3f} (Δ = {min_alpha - baseline:+.3f})")
    print(f"  Spearman r:         {spearman_r:+.3f} (p = {spearman_p:.4f})")
    print(f"  Pearson r:          {pearson_r:+.3f} (p = {pearson_p:.4f})")
    
    # Interpretation
    if spearman_p < 0.05 and spearman_r > 0.3:
        print(f"  → ✓ SIGNIFICANT POSITIVE TREND")
    elif spearman_p < 0.05 and spearman_r < -0.3:
        print(f"  → ✗ SIGNIFICANT NEGATIVE TREND (unexpected!)")
    else:
        print(f"  → ○ NO SIGNIFICANT TREND (flat)")

print("\n" + "="*70)

## 12. Save Results

In [ ]:
# Save full results
results_df.to_csv(RESULTS_CSV, index=False)
print(f"Saved full results to {RESULTS_CSV}")

# Save summary
summary.to_csv(SUMMARY_CSV, index=False)
print(f"Saved summary to {SUMMARY_CSV}")

print(f"\nTotal results: {len(results_df)}")
print(f"Total conditions: {len(summary)}")

## 13. Interpretation Guide

### ✅ Hypothesis SUPPORTED if:
- **Positive correlation:** Acceptance increases with α across scenarios
- **Effect size:** Δ(α=+4 vs α=0) > 0.2 for most scenarios
- **Control:** Random vector shows flat/weak trend
- **Interpretation:** Surface vector causally affects acceptance → thought injection works

### ❌ Hypothesis REJECTED if:
- **Flat trend:** No correlation between α and acceptance
- **Small effect:** |Δ| < 0.1 across all scenarios
- **Interpretation:** Vector is probe-only, doesn't enable thought injection

### ⚠️ Mixed Results if:
- **Scenario-dependent:** Works for some tasks, not others
- **Weak effect:** 0.1 < Δ < 0.2 (detectable but not practical)
- **Interpretation:** Vector has some causal effect but insufficient to override priors

### 🔄 Unexpected Results if:
- **Negative correlation:** Acceptance decreases with α (sign error?)
- **Random = Surface:** Control vector performs same as surface vector
- **Interpretation:** Something else is going on (steering artifact, model instability)

---

**Next Steps Based on Results:**

1. **If REJECTED:** Re-extract vector using single-sentence diffs (not full-trace paraphrasing)
2. **If SUPPORTED:** Proceed to test with naturally-written injections (not paraphrased)
3. **If MIXED:** Investigate which scenarios work and why (prior strength? task type?)
4. **If UNEXPECTED:** Debug vector extraction or steering implementation